In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

### EDA

In [2]:
df = pd.read_csv("../../datasets/dataset/train.csv")

In [3]:
print("length of dataset: ", len(df))
print("Labels distribution: ", df['label'].value_counts(normalize=True) * 100)
print("Total missing values: ", df.isnull().sum().sum())

length of dataset:  40000
Labels distribution:  label
0    96.8
1     3.2
Name: proportion, dtype: float64
Total missing values:  153603


In [4]:
from pandas.core.arrays import categorical


dense_cols = [f"integer_feature_{i}" for i in range(1, 14)]
categorical_cols = [f"categorical_feature_{i}" for i in range(1, 27)]

df[dense_cols] = df[dense_cols].fillna(0)

for col in dense_cols:
    df[col] = np.log(1 + df[col] - df[col].min() + 1)

df[dense_cols] = MinMaxScaler().fit_transform(df[dense_cols])

df[categorical_cols] = df[categorical_cols].fillna('UNK')
label_encoders_dict = {}
sparse_cardinalities = []

for col in categorical_cols:
    l_e = LabelEncoder()
    df[col] = l_e.fit_transform(df[col].astype(str))
    label_encoders_dict[col] = l_e  # <
    sparse_cardinalities.append(df[col].nunique())

    

In [5]:
class customDataset(Dataset):
    def __init__(self, data):
        self.dense = torch.tensor(data[dense_cols].values, dtype=torch.float32)
        self.categorical = torch.tensor(data[categorical_cols].values, dtype=torch.long)
        self.labels = torch.tensor(data['label'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.dense[index], self.categorical[index], self.labels[index]

In [6]:

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader = DataLoader(customDataset(train_df), batch_size=1024, shuffle=True)
val_loader = DataLoader(customDataset(val_df), batch_size=1024, shuffle=False)


In [38]:
class VanillaCTR(nn.Module):
    def __init__(self, sparse_cardinalities, num_dense_features, embed_dim=16, hidden_dims=[256, 128, 64], dropout=0.3):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=int(vocab_size), embedding_dim=embed_dim) for vocab_size in sparse_cardinalities
        ])

        total_input_dim = (len(sparse_cardinalities) * embed_dim) + num_dense_features

        layers = []
        input_dim = total_input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, dense_x, categ_x):
        embeds = [
            self.embeddings[i](categ_x[:, i]) for i in range(categ_x.shape[1])   #*
        ]

        embeds_concat = torch.cat(embeds, dim=1)

        x = torch.cat([dense_x,  embeds_concat], dim=1)

        out = self.mlp(x)

        return out.squeeze()




    



In [117]:
class DeepFMCTR(nn.Module):
    def __init__(self, sparse_cardinalities, embed_dim=8, num_dense_features=13, hidden_dims = [128, 128, 64], dropout=0.4):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=int(sz), embedding_dim=embed_dim) for sz in sparse_cardinalities
        ])

        total_input_size = num_dense_features + (len(sparse_cardinalities) * embed_dim)

        layers = []
        input_dim = total_input_size
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU(hidden_dim))
            layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim
            
        self.mlp = nn.Sequential(*layers)
        self.deep_out = nn.Linear(hidden_dims[-1], 1)

        self.dense_linear = nn.Linear(num_dense_features, 1)

    def forward(self, dense_x, categ_x):
        embed = [
            self.embeddings[i](categ_x[:, i]) for i in range(categ_x.shape[1])
        ]

        embeds_stacked = torch.stack(embed, dim=1)

        embeds_sum = torch.sum(embeds_stacked, dim = 1)
        sum_sqr = torch.pow(embeds_sum, 2)

        embeds_sqr = torch.pow(embeds_stacked, 2)
        sqr_sum = torch.sum(embeds_sqr, dim = 1)

        fm_out = 0.5*torch.sum(sum_sqr - sqr_sum, dim = 1, keepdim=True)

        embeds_concat = torch.cat(embed, dim = 1)
        x_0 = torch.cat([dense_x, embeds_concat], dim=1)
        deep_pred = self.deep_out(self.mlp(x_0))

        dense_pred = self.dense_linear(dense_x)

        final_out = fm_out + deep_pred + dense_pred

        return final_out.squeeze()
        





In [118]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model1 = VanillaCTR(
    sparse_cardinalities=sparse_cardinalities, 
    num_dense_features=len(dense_cols)
).to(device)

print(model1)

model2 = DeepFMCTR(
    sparse_cardinalities=sparse_cardinalities, 
    num_dense_features=len(dense_cols)
).to(device)

print(model2)

VanillaCTR(
  (embeddings): ModuleList(
    (0): Embedding(13191, 16)
    (1): Embedding(3757, 16)
    (2): Embedding(4968, 16)
    (3): Embedding(1851, 16)
    (4): Embedding(4253, 16)
    (5): Embedding(3, 16)
    (6): Embedding(3111, 16)
    (7): Embedding(741, 16)
    (8): Embedding(22, 16)
    (9): Embedding(11585, 16)
    (10): Embedding(5569, 16)
    (11): Embedding(7723, 16)
    (12): Embedding(9, 16)
    (13): Embedding(904, 16)
    (14): Embedding(2025, 16)
    (15): Embedding(43, 16)
    (16): Embedding(4, 16)
    (17): Embedding(341, 16)
    (18): Embedding(14, 16)
    (19): Embedding(13509, 16)
    (20): Embedding(9003, 16)
    (21): Embedding(12629, 16)
    (22): Embedding(5095, 16)
    (23): Embedding(3504, 16)
    (24): Embedding(36, 16)
    (25): Embedding(24, 16)
  )
  (mlp): Sequential(
    (0): Linear(in_features=429, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropou

In [119]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score, log_loss, f1_score

def train(model, lr, weight_decay, pos_weight_val, epochs=20):
    pos_weight = torch.tensor([pos_weight_val]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.AdamW(model.parameters(), lr = lr, weight_decay=weight_decay)

    training_losses = []
    val_losses = []

    # begin training

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0

        for dense_x, categ_x, labels in train_loader:
            dense_x, categ_x, labels = dense_x.to(device), categ_x.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(dense_x, categ_x)
            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * labels.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        training_losses.append(epoch_loss)

        model.eval()
        running_loss = 0.

        all_preds = []

        all_labels = []

        with torch.no_grad():
            for dense_x, sparse_x, labels in val_loader:
                dense_x, sparse_x, labels = dense_x.to(device), sparse_x.to(device), labels.to(device)
                out = model(dense_x, sparse_x)

                loss = criterion(out, labels)

                running_loss += loss.item() * labels.size(0)

                probs = torch.sigmoid(out)
                all_preds.extend(probs.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
        epoch_valid_loss = running_loss / len(val_loader.dataset)
        val_losses.append(epoch_valid_loss)

        print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_valid_loss:.4f}")

    return all_preds, all_labels

In [120]:
import optuna
import numpy as np

def objective(trial):
    # 1. Let Optuna pick the hyperparameters for this specific run
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout", 0.1, 0.5)
    pw = trial.suggest_float("pos_weight", 5.0, 25.0) 
    
    # 2. Initialize the model with Optuna's chosen dropout
    model = DeepFMCTR(
        sparse_cardinalities=sparse_cardinalities,
        num_dense_features=len(dense_cols),
        embed_dim=32, 
        dropout=dropout_rate
    ).to(device)
    
    # 3. Call your updated train function 
    # (Keep epochs at 2 or 3 so the 20-trial sweep finishes quickly)
    all_preds, all_labels = train(model, lr, weight_decay, pw, epochs=3)
    
    # 4. Calculate ROC-AUC so Optuna knows how well this combination performed
    all_labels = np.array(all_labels).astype(int)
    roc_auc = roc_auc_score(all_labels, all_preds)
    
    return roc_auc

# --- Execute the Sweep ---
study = optuna.create_study(direction="maximize")
print("Starting Optuna Hyperparameter Sweep...")
study.optimize(objective, n_trials=20)

[I 2026-10-02 14:46:11,266] A new study created in memory with name: no-name-12a59119-4e56-4534-9cff-df27e7b32932


Starting Optuna Hyperparameter Sweep...
Epoch 01/3 | Train Loss: 50.8396 | Val Loss: 40.1127
Epoch 02/3 | Train Loss: 32.0707 | Val Loss: 34.7560


[I 2026-10-02 14:46:13,595] Trial 0 finished with value: 0.5199572350368027 and parameters: {'lr': 0.0020222195430348202, 'weight_decay': 1.7198615390829006e-06, 'dropout': 0.1756427459373453, 'pos_weight': 11.049114131971896}. Best is trial 0 with value: 0.5199572350368027.


Epoch 03/3 | Train Loss: 24.2368 | Val Loss: 32.7933
Epoch 01/3 | Train Loss: 60.1775 | Val Loss: 50.2187
Epoch 02/3 | Train Loss: 31.4231 | Val Loss: 45.9720


[I 2026-10-02 14:46:16,057] Trial 1 finished with value: 0.5677303597946797 and parameters: {'lr': 0.005793284791218981, 'weight_decay': 1.1387926881043743e-06, 'dropout': 0.43518594202114513, 'pos_weight': 23.51321521965981}. Best is trial 1 with value: 0.5677303597946797.


...

Epoch 03/3 | Train Loss: 8.2379 | Val Loss: 47.7129
Epoch 01/3 | Train Loss: 59.6304 | Val Loss: 56.1949
Epoch 02/3 | Train Loss: 44.7433 | Val Loss: 52.8875


[I 2026-10-02 14:46:54,440] Trial 18 finished with value: 0.5157175615799328 and parameters: {'lr': 0.0021418220172699023, 'weight_decay': 3.328301391539147e-06, 'dropout': 0.11293220116028574, 'pos_weight': 22.923778772796}. Best is trial 15 with value: 0.5893332741477273.


Epoch 03/3 | Train Loss: 33.1451 | Val Loss: 49.7253
Epoch 01/3 | Train Loss: 59.8768 | Val Loss: 54.1004
Epoch 02/3 | Train Loss: 39.7770 | Val Loss: 49.6245


[I 2026-10-02 14:46:56,836] Trial 19 finished with value: 0.5650952551975722 and parameters: {'lr': 0.0029438336026068545, 'weight_decay': 0.0001106342474370171, 'dropout': 0.22496163424512008, 'pos_weight': 23.33189950085863}. Best is trial 15 with value: 0.5893332741477273.


Epoch 03/3 | Train Loss: 26.4891 | Val Loss: 47.6063


In [121]:
print("\n" + "="*30)
print(f"Best ROC-AUC Achieved: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


Best ROC-AUC Achieved: 0.5893
Best Hyperparameters:
  lr: 0.002527040403217253
  weight_decay: 3.161801387597611e-05
  dropout: 0.10902412128805176
  pos_weight: 21.820622345166857


## Evaluations

In [115]:
all_preds1, all_labels1 = train(model1)
all_labels1 = np.array(all_labels1).astype(int)
binary_predxns1 = [1 if p >= 0.6 else 0 for p in all_preds1]

roc_auc = roc_auc_score(all_labels1, all_preds1)
pr_auc = average_precision_score(all_labels1, all_preds1)
accuracy = accuracy_score(all_labels1, binary_predxns1)
logloss = log_loss(all_labels1, all_preds1)
f1 = f1_score(all_labels1, binary_predxns1)

print("ROC-AUC: ", roc_auc)
print("Precision Recall: ", pr_auc)
print("Accuracy: ", accuracy)
print("logloss: ", logloss)
print("f1 Scoe: ", f1)

TypeError: train() missing 3 required positional arguments: 'lr', 'weight_decay', and 'pos_weight_val'

In [ ]:
all_preds2, all_labels2 = train(model2)
all_labels2 = np.array(all_labels2).astype(int)
binary_predxns2 = [1 if p >= 0.3 else 0 for p in all_preds2]

roc_auc = roc_auc_score(all_labels2, all_preds2)
pr_auc = average_precision_score(all_labels2, all_preds2)
accuracy = accuracy_score(all_labels2, binary_predxns2)
logloss = log_loss(all_labels2, all_preds2)
f1 = f1_score(all_labels2, binary_predxns2)

print("ROC-AUC: ", roc_auc)
print("Precision Recall: ", pr_auc)
print("Accuracy: ", accuracy)
print("logloss: ", logloss)
print("f1 Scoe: ", f1)

Epoch 01/20 | Train Loss: 0.5363 | Val Loss: 15.4390
Epoch 02/20 | Train Loss: 0.4633 | Val Loss: 15.6651
Epoch 03/20 | Train Loss: 0.3863 | Val Loss: 15.4226
Epoch 04/20 | Train Loss: 0.3250 | Val Loss: 15.4518
Epoch 05/20 | Train Loss: 0.3271 | Val Loss: 15.5897
Epoch 06/20 | Train Loss: 0.2667 | Val Loss: 15.6183
Epoch 07/20 | Train Loss: 0.2359 | Val Loss: 15.5681
Epoch 08/20 | Train Loss: 0.2375 | Val Loss: 15.5852
Epoch 09/20 | Train Loss: 0.1371 | Val Loss: 15.6376
Epoch 10/20 | Train Loss: 0.1502 | Val Loss: 15.8993
Epoch 11/20 | Train Loss: 0.1498 | Val Loss: 15.3567
Epoch 12/20 | Train Loss: 0.1462 | Val Loss: 15.5521
Epoch 13/20 | Train Loss: 0.1075 | Val Loss: 15.4851
Epoch 14/20 | Train Loss: 0.1182 | Val Loss: 15.3531
Epoch 15/20 | Train Loss: 0.0947 | Val Loss: 15.5603
Epoch 16/20 | Train Loss: 0.0958 | Val Loss: 15.7320
Epoch 17/20 | Train Loss: 0.0678 | Val Loss: 15.6886
Epoch 18/20 | Train Loss: 0.0602 | Val Loss: 15.5751
Epoch 19/20 | Train Loss: 0.0545 | Val Loss: 1

In [122]:
import numpy as np
from sklearn.metrics import recall_score, precision_score

# 1. Initialize final model with Optuna's best dropout
final_model = DeepFMCTR(
    sparse_cardinalities=sparse_cardinalities,
    num_dense_features=len(dense_cols),
    embed_dim=32, 
    dropout=0.25
).to(device)

# 2. Train for a full 20 epochs using Optuna's best parameters
print("Training final DeepFM model...")
all_preds, all_labels = train(
    final_model, 
    lr=0.008, 
    weight_decay=8e-4,
    pos_weight_val=16.76,
    epochs=20
)

# 3. Calculate Final ROC-AUC
all_labels = np.array(all_labels).astype(int)
final_roc_auc = roc_auc_score(all_labels, all_preds)
print("\n" + "="*40)
print(f"Final Model ROC-AUC: {final_roc_auc:.4f}")

# 4. Sweep for the Optimal F1 Threshold
best_f1, best_thresh, best_acc, best_rec, best_prec = 0.0, 0.5, 0.0, 0.0, 0.0

for thresh in np.arange(0.10, 0.90, 0.01):
    temp_preds = [1 if p >= thresh else 0 for p in all_preds]
    f1 = f1_score(all_labels, temp_preds, zero_division=0)
    
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh
        best_acc = accuracy_score(all_labels, temp_preds)
        best_rec = recall_score(all_labels, temp_preds, zero_division=0)
        best_prec = precision_score(all_labels, temp_preds, zero_division=0)

print(f"Optimal F1 Threshold: {best_thresh:.2f}")
print(f"Maximized F1:         {best_f1:.4f}")
print(f"Recall at Threshold:  {best_rec:.4f} (Caught {best_rec*100:.1f}% of real clicks)")
print(f"Precision at Thresh:  {best_prec:.4f}")
print(f"Accuracy at Thresh:   {best_acc:.4f}")
print("="*40)

Training final DeepFM model...
Epoch 01/20 | Train Loss: 47.8561 | Val Loss: 44.1505
Epoch 02/20 | Train Loss: 19.9690 | Val Loss: 41.9945
Epoch 03/20 | Train Loss: 8.6653 | Val Loss: 41.2532
Epoch 04/20 | Train Loss: 4.0582 | Val Loss: 40.3212
Epoch 05/20 | Train Loss: 2.3429 | Val Loss: 40.6286
Epoch 06/20 | Train Loss: 1.5327 | Val Loss: 40.7407
Epoch 07/20 | Train Loss: 0.9749 | Val Loss: 40.9824
Epoch 08/20 | Train Loss: 0.7360 | Val Loss: 41.0375
Epoch 09/20 | Train Loss: 0.5845 | Val Loss: 40.9344
Epoch 10/20 | Train Loss: 0.4007 | Val Loss: 40.8577
Epoch 11/20 | Train Loss: 0.3508 | Val Loss: 41.3260
Epoch 12/20 | Train Loss: 0.2325 | Val Loss: 40.9092
Epoch 13/20 | Train Loss: 0.2112 | Val Loss: 42.0468
Epoch 14/20 | Train Loss: 0.1784 | Val Loss: 41.4250
Epoch 15/20 | Train Loss: 0.1457 | Val Loss: 41.2180
Epoch 16/20 | Train Loss: 0.1173 | Val Loss: 40.8951
Epoch 17/20 | Train Loss: 0.1083 | Val Loss: 41.8126
Epoch 18/20 | Train Loss: 0.1241 | Val Loss: 41.8070
Epoch 19/20 |